# 🧪 Creating Custom Gymnasium Environments
This tutorial covers how to build and test custom Gymnasium environments for Reinforcement Learning (RL). We'll begin with a simple 1D movement environment, then move to a more complex example simulating an Emergency Room in a hospital.

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This section performs environment registration or additional logic.


In [ ]:

# Activate the python environment   as we did install it in the previous lab 
#source rl-env/bin/activate  # On Windows: rl-env\Scripts\activate

In [ ]:
# ✅ Install gymnasium if not already installed
#!pip install gymnasium numpy
# if it is already installed, it will skip the installation step
import gymnasium as gym
import numpy as np

## 📦 SimpleEnv: 1D Navigation to a Goal
- Action space: 0 = left, 1 = right
- State space: 1D position from 0 to 10
- Goal: Reach position >= 10

![Simple ENV](./images/simple.png)

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This defines a new custom environment class inheriting from `gym.Env`.
- It sets up the action space, observation space, and environment metadata.


In [ ]:
# Import the Gymnasium library for building custom environments
import gymnasium as gym

# Import the 'spaces' module to define action and observation spaces
from gymnasium import spaces

# Import NumPy for numerical operations
import numpy as np

# Define a custom environment by inheriting from gym.Env
class SimpleEnv(gym.Env):
    # Optional metadata specifying available render modes
    metadata = {"render_modes": ["human"]}

    def __init__(self):
        # Call the constructor of the base class
        super(SimpleEnv, self).__init__()

        # Define the action space:
        # Discrete(2) means two actions: 0 (left), 1 (right)
        self.action_space = spaces.Discrete(2)

        # Define the observation (state) space:
        # A 1D position in the range [0, 10]
        self.observation_space = spaces.Box(low=0, high=10, shape=(1,), dtype=np.float32)

        # Set the goal position the agent should reach
        self.goal_position = 10.0

    def reset(self, seed=None, options=None):
        # Reset the environment state at the beginning of an episode
        # Optionally use a seed for reproducibility
        super().reset(seed=seed)

        # Initialize the agent's position randomly between 0 and 5
        self.state = np.array([np.random.uniform(0, 5)], dtype=np.float32)

        # Return the initial state and empty info dictionary
        return self.state, {}

    def step(self, action):
        # Apply the action:
        # If 0, move left (decrease position by 1)
        # If 1, move right (increase position by 1)
        if action == 0:
            self.state[0] -= 1.0
        elif action == 1:
            self.state[0] += 1.0

        # Ensure the state remains within the defined boundaries [0, 10]
        self.state = np.clip(self.state, 0.0, 10.0)

        # Check if the goal has been reached
        done = self.state[0] >= self.goal_position

        # Provide reward:
        # +10 for reaching the goal, -1 for each step otherwise
        reward = 10.0 if done else -1.0

        # Return new state, reward, done flag, truncated flag, and info
        return self.state, reward, done, False, {}

    def render(self):
        # Display the current position of the agent
        print(f"Agent Position: {self.state[0]}")

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This line creates an instance of the environment and simulates it using random actions.


In [ ]:
# Run SimpleEnv simulation
env = SimpleEnv()
obs, _ = env.reset()
done = False
while not done:
    action = env.action_space.sample()
    obs, reward, done, _, _ = env.step(action)
    env.render()

## 🏥 EmergencyRoomEnv: Simulating Patient Treatment
**Action Space:** `0 = treat patient`, `1 = wait`

**Observation Space:** `[patients_waiting, doctor_available]`

**Goal:** Treat all patients efficiently. Penalty for waiting too long or idle time.

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This defines a new custom environment class inheriting from `gym.Env`.
- It sets up the action space, observation space, and environment metadata.


In [ ]:
# Define a custom Gymnasium environment simulating an emergency room
class EmergencyRoomEnv(gym.Env):
    
    # Constructor to initialize the environment
    def __init__(self):
        super().__init__()  # Call the base class constructor

        # Define the action space:
        # Two possible actions: 0 = treat a patient, 1 = wait
        self.action_space = spaces.Discrete(2)

        # Define the observation space:
        # A Box space with two integers: [patients_waiting, doctor_available]
        # Patients: 0 to 10 (you can adjust), Doctor availability: 0 or 1
        self.observation_space = spaces.Box(low=0, high=10, shape=(2,), dtype=np.int32)

        # Initialize environment state using the reset function
        self.reset()

    # Reset the environment to the initial state
    def reset(self, seed=None, options=None):
        # Start with 1000 patients to treat
        self.patients = 1000

        # The doctor is available at the start (1 means available)
        self.doctor_available = 1

        # Time counter starts from 0
        self.time = 0

        # Return the initial observation and empty info dict
        return np.array([self.patients, self.doctor_available]), {}

    # Define how the environment responds to an action
    def step(self, action):
        reward = 0  # Initialize reward

        # Action 0: Treat a patient (if possible)
        if action == 0 and self.patients > 0 and self.doctor_available:
            self.patients -= 1  # One patient is treated
            reward = 10         # Reward for successful treatment

        # Action 1: Wait (no treatment)
        elif action == 1:
            reward = -2         # Small penalty for doing nothing

        # Invalid action or no patients/doctor available
        else:
            reward = -5         # Heavy penalty for invalid action

        # Increase time step
        self.time += 1

        # Episode is done if all patients are treated or time runs out
        done = self.patients == 0 or self.time >= 20

        # Return updated state, reward, done flag, False (truncated), and empty info dict
        return np.array([self.patients, self.doctor_available]), reward, done, False, {}

    # Render the environment state for human observation
    def render(self):
        print(f"Time: {self.time}, Patients: {self.patients}, Doctor: {self.doctor_available}")

# 🧠 Explanation of the next code cell
This code demonstrates one of the core components of a Gymnasium environment.

- This line creates an instance of the environment and simulates it using random actions.


In [ ]:
# Run EmergencyRoomEnv simulation
env = EmergencyRoomEnv()
obs, _ = env.reset()
done = False
total_reward = 0
while not done:
    action = env.action_space.sample()
    obs, reward, done, _, _ = env.step(action)
    total_reward += reward
    env.render()
print(f"Total reward: {total_reward}")